# Run GEE Spatial Extractions

Use this notebook as the Colab runner. The reusable workflow code lives in the repo.

Before running, upload the zipped watershed shapefile listed in `config/gee-assets.yml` to Earth Engine using the asset name in that same config file. The first code cell clones the repo if needed, or refreshes `/content/data-workflow_spatial` so stale code does not get reused. The active run is set in `config/run-list.yml`; the notebook builds the full run list and launches only the configured chunk for that session.

The current active run is set in `config/run-list.yml`. The active annual ERA5-Land run starts in 2000; the 2001-2022 window is only for cross-product comparison runs. Exports go to the destination listed in `config/gee-assets.yml`.

Earth Engine shortens some shapefile column names during upload. This notebook checks for the shortened `run_grp` field before filtering runs.


In [ ]:
REPO_URL = "https://github.com/global-river-chem/data-workflow_spatial.git"
REPO_DIR = "/content/data-workflow_spatial"

import os
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

import sys
import importlib

for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src.gee_spatial"):
        del sys.modules[module_name]
importlib.invalidate_caches()


In [ ]:
!pip -q install -r requirements.txt

In [ ]:
from src.gee_spatial.auth import initialize
from src.gee_spatial.extraction import load_run_config

assets = load_run_config("config/gee-assets.yml")
products = load_run_config("config/driver-products.yml")
run_list = load_run_config("config/run-list.yml")

ee = initialize(project=assets.get("earth_engine", {}).get("project"))

In [ ]:
from src.gee_spatial.datasets import continuous_products, product_names
from src.gee_spatial.extraction import load_watersheds
from src.gee_spatial.runs import all_run_groups, choose_property_name

watersheds = load_watersheds(assets["watersheds"]["asset_id"])
configured_run_group_column = assets.get("watersheds", {}).get(
    "earth_engine_run_group_column",
    run_list.get("site_groups", {}).get("column", "run_group"),
)
run_group_column = choose_property_name(
    watersheds,
    configured_run_group_column,
    alternatives=["run_group", "run_grp"],
)
available_run_groups = all_run_groups(watersheds, run_group_column)
print("Products:", product_names(products))
print("Continuous products:", continuous_products(products))
print("Run group column:", run_group_column)
print("Asset rows:", watersheds.size().getInfo())
print("Run groups:", len(available_run_groups), available_run_groups[:10])
print("Run group counts:", watersheds.aggregate_histogram(run_group_column).getInfo())
print(watersheds.limit(1).getInfo())


In [ ]:
from pathlib import Path

from src.gee_spatial.export import export_table, task_summary
from src.gee_spatial.extraction import (
    EXPORT_COLUMNS,
    era5_export_columns,
    extract_continuous_product,
    extract_era5_land_monthly_year_products,
    extract_era5_land_products,
)
from src.gee_spatial.runs import build_run_list, export_name, filter_watersheds_by_group, run_list_chunk
from src.gee_spatial.timing import (
    datetime_label,
    print_timing_summary,
    task_timing_row,
    timing_rows_from_launched_tasks,
    utc_now,
    write_timing_log,
)

run_rows = build_run_list(run_list, available_run_groups)
launch_settings = run_list.get("launch", {})
start_at = int(launch_settings.get("start_at", 0))
max_tasks = launch_settings.get("max_tasks_per_session", 5)
WAIT_FOR_TASKS = bool(launch_settings.get("wait_for_tasks", False))
rows_to_launch = run_list_chunk(run_rows, start_at=start_at, max_tasks=max_tasks)
active_run = run_list.get("active_run")
run_settings = (run_list.get("runs") or {}).get(active_run, {})
run_export = bool(run_settings.get("launch_export", False))
session_started_at = utc_now()
timing_log_path = Path("timing-logs") / f"gee_task_timing_{active_run}_{datetime_label(session_started_at)}.csv"

print("Active run:", active_run)
print("Total planned tasks:", len(run_rows))
print("Tasks in this chunk:", len(rows_to_launch))
print("Launch export:", run_export)
print("Wait for tasks:", WAIT_FOR_TASKS)
print("Timing log:", timing_log_path)

launched_tasks = []

for row_number, run_row in enumerate(rows_to_launch, start=start_at + 1):
    MODE = run_row.get("mode", "single_product")
    PRODUCT = run_row.get("product")
    PRODUCTS = run_row.get("products")
    YEAR = run_row.get("year")
    MONTH = run_row.get("month")
    MONTHS = run_row.get("months")
    PERIOD = run_row.get("period")
    RUN_GROUP = run_row.get("run_group")

    selected_watersheds = filter_watersheds_by_group(
        watersheds,
        run_group=RUN_GROUP,
        run_group_column=run_group_column,
    )
    selected_row_count = selected_watersheds.size().getInfo()
    export_label = MODE if MODE == "era5_land" else PRODUCT
    out_name = export_name(export_label, year=YEAR, month=MONTH, run_group=RUN_GROUP, period=PERIOD)

    print("\nTask", row_number, "of", len(run_rows))
    print("Planned export:", out_name)
    print("Selected rows:", selected_row_count)

    if selected_row_count == 0:
        raise ValueError(
            f"No watershed rows selected for {RUN_GROUP} using column {run_group_column}. "
            "Check the uploaded asset column names before exporting."
        )

    if not run_export:
        continue

    if MODE == "era5_land":
        if PERIOD == "monthly_by_year":
            export_rows = extract_era5_land_monthly_year_products(
                products,
                selected_watersheds,
                year=YEAR,
                months=MONTHS,
                product_names=PRODUCTS,
            )
        else:
            export_rows = extract_era5_land_products(
                products,
                selected_watersheds,
                year=YEAR,
                month=MONTH,
                product_names=PRODUCTS,
            )
        selectors = era5_export_columns(products, product_names=PRODUCTS)
    else:
        export_rows = extract_continuous_product(
            PRODUCT,
            products,
            selected_watersheds,
            year=YEAR,
            month=MONTH,
        )
        selectors = EXPORT_COLUMNS

    task = export_table(
        export_rows,
        description=out_name,
        export_config=assets["exports"],
        file_name_prefix=out_name,
        selectors=selectors,
    )
    launched_at = utc_now()
    timing_row = task_timing_row(
        run_row=run_row,
        export_name=out_name,
        task=task,
        selected_rows=selected_row_count,
        export_destination=assets["exports"].get("destination", "drive"),
        launched_at=launched_at,
    )
    launched_tasks.append({"name": out_name, "task": task, "launched_at": launched_at, "timing_row": timing_row})
    write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)
    print(task_summary(task))


In [ ]:
import time

from src.gee_spatial.timing import (
    DONE_STATES,
    FAILED_STATES,
    print_timing_summary,
    timing_rows_from_launched_tasks,
    update_task_timing_row,
    utc_now,
    write_timing_log,
)

poll_seconds = 60
wait_for_tasks = bool(globals().get("WAIT_FOR_TASKS", False))


def print_existing_earth_engine_tasks(name_filter=None, max_tasks=40):
    name_filter = name_filter or ""
    matches = []
    for task in ee.batch.Task.list():
        status = task.status()
        description = status.get("description", "")
        if name_filter and name_filter not in description:
            continue
        matches.append(status)

    if not matches:
        print("No Earth Engine tasks matched filter:", name_filter or "<none>")
        return

    print("Earth Engine task status snapshot for filter:", name_filter or "<none>")
    for status in matches[:max_tasks]:
        print(status.get("description", ""), status.get("state", ""), status.get("error_message", ""))
    if len(matches) > max_tasks:
        print("Additional matching tasks not shown:", len(matches) - max_tasks)


def default_task_status_filter():
    if "RUN_LTER_LABEL" in globals() and "RUN_LABEL" in globals():
        return f"{RUN_LTER_LABEL}_{RUN_LABEL}"
    return globals().get("RUN_LABEL", globals().get("active_run", ""))

def refresh_launched_tasks_once():
    now = utc_now()
    print()
    print("Task status snapshot at", now.isoformat(timespec="seconds"))
    all_done = True

    for item in launched_tasks:
        item["timing_row"] = update_task_timing_row(
            item["timing_row"],
            item["task"],
            checked_at=now,
        )
        state = item["timing_row"].get("state")
        elapsed_min = float(item["timing_row"].get("elapsed_min") or 0)
        error_message = item["timing_row"].get("error_message", "")
        print(f"{item['name']}: {state}, elapsed {elapsed_min:.1f} min")
        if error_message:
            print("  error:", error_message)

        if state not in DONE_STATES:
            all_done = False

    write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)
    return all_done


if "launched_tasks" not in globals() or not launched_tasks:
    print("No tasks were launched in this Colab session memory.")
    print_existing_earth_engine_tasks(default_task_status_filter())
else:
    all_done = refresh_launched_tasks_once()

    if wait_for_tasks:
        while not all_done:
            time.sleep(poll_seconds)
            all_done = refresh_launched_tasks_once()
    else:
        print()
        print("WAIT_FOR_TASKS is FALSE, so this notebook will not keep polling.")
        print("The Earth Engine exports continue server-side after launch.")
        print("Rerun this cell later, or check the Earth Engine task list, to see final status.")

    final_rows = timing_rows_from_launched_tasks(launched_tasks)
    print_timing_summary(final_rows)
    failed_rows = [row for row in final_rows if row.get("state") in FAILED_STATES]

    run_wall_clock_finished_at = utc_now()
    run_wall_clock_elapsed_min = None
    if "run_wall_clock_timer_started" in globals():
        run_wall_clock_elapsed_min = (time.perf_counter() - run_wall_clock_timer_started) / 60
    elif "run_wall_clock_started_at" in globals():
        run_wall_clock_elapsed_min = (
            run_wall_clock_finished_at - run_wall_clock_started_at
        ).total_seconds() / 60

    if failed_rows:
        if "write_run_wall_clock_summary" in globals():
            write_run_wall_clock_summary(
                status="failed",
                finished_at=run_wall_clock_finished_at,
                elapsed_min=run_wall_clock_elapsed_min,
            )
        for row in failed_rows:
            print(row.get("export_name"), row.get("state"), row.get("error_message", ""))
        raise RuntimeError(f"Earth Engine export failed for {len(failed_rows)} task(s).")

    if wait_for_tasks and all_done:
        summary_status = "completed"
    elif all_done:
        summary_status = "completed_snapshot"
    else:
        summary_status = "launched"

    if "write_run_wall_clock_summary" in globals():
        write_run_wall_clock_summary(
            status=summary_status,
            finished_at=run_wall_clock_finished_at,
            elapsed_min=run_wall_clock_elapsed_min,
        )
        print("Run timing summary:", run_timing_summary_path)

    print("Task timing log:", timing_log_path)


In [ ]:
# Optional: rerun this cell later to check matching Earth Engine tasks after reconnecting.
TASK_STATUS_FILTER = default_task_status_filter() if "default_task_status_filter" in globals() else globals().get("RUN_LABEL", globals().get("active_run", ""))

if "print_existing_earth_engine_tasks" not in globals():
    def print_existing_earth_engine_tasks(name_filter=None, max_tasks=40):
        name_filter = name_filter or ""
        matches = []
        for task in ee.batch.Task.list():
            status = task.status()
            description = status.get("description", "")
            if name_filter and name_filter not in description:
                continue
            matches.append(status)

        if not matches:
            print("No Earth Engine tasks matched filter:", name_filter or "<none>")
            return

        print("Earth Engine task status snapshot for filter:", name_filter or "<none>")
        for status in matches[:max_tasks]:
            print(status.get("description", ""), status.get("state", ""), status.get("error_message", ""))
        if len(matches) > max_tasks:
            print("Additional matching tasks not shown:", len(matches) - max_tasks)

print_existing_earth_engine_tasks(TASK_STATUS_FILTER)
